<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 02 · Observe actual Spark scans</h1><p>SDA-DSC-214 · Meaad Al-Marri</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اللاب 02 · لاحظ قياس Spark الفعلي</h1><p>SDA-DSC-214 · ميعاد المري</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Goal and status</h2><p><strong>ENGINE_NOT_EXECUTED</strong> — no measured timings are saved. Read <a href="../../labs/lab02/WALKTHROUGH.md">the walkthrough</a>. This notebook requires successful Bronze, compares equal 72-row populations and makes no speed-up promise.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الهدف والحالة</h2><p><strong>ENGINE_NOT_EXECUTED</strong> — لا توجد أزمنة مقاسة محفوظة. اقرأ <a href="../../labs/lab02/WALKTHROUGH.md">الشرح المتدرج</a>. يتطلب الدفتر Bronze ناجحة ويقارن عينتين متطابقتين من 72 صفًا، ولا يعد بتسارع محدد.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Setup</h2><p>Inspect the actual environment before accessing a prior workspace.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الإعداد</h2><p>افحص البيئة الفعلية قبل الوصول إلى مساحة عمل سابقة.</p></td></tr></tbody></table>

In [15]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

{
  "scope": "DEPENDENCY_PREFLIGHT_ONLY",
  "python": "3.11.16",
  "java": "openjdk version \"17.0.20.1\" 2026-08-18",
  "java_major": 17,
  "packages": {
    "pyspark": {
      "required": "3.5.8",
      "observed": "3.5.8"
    },
    "delta-spark": {
      "required": "3.3.2",
      "observed": "3.3.2"
    },
    "py4j": {
      "required": "0.10.9.9",
      "observed": "0.10.9.9"
    }
  },
  "status": "DEPENDENCIES_PRESENT_ENGINE_NOT_TESTED",
  "issues": [],
  "engine_executed": false
}


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Reuse the successful project state</h2><p>No source regeneration, table overwrite or new independent project.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>استخدم حالة المشروع الناجحة</h2><p>لا إعادة توليد للمصدر ولا استبدال للجداول ولا مشروع جديد مستقل.</p></td></tr></tbody></table>

In [16]:
require_environment()
from masar.workspace import completed_bronze_workspace, require_fixed_dataset
require_fixed_dataset(SOURCE)
WORK = completed_bronze_workspace(ROOT)
spark = start_spark(WORK)
print("Workspace:", WORK.relative_to(ROOT))

Workspace: outputs/day01_bronze_n6_lk_hh


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Measure equal-result actions</h2><p>CSV versus Delta version 0; one warm-up each, four measurements each in balanced order. Shared code retains the raw measurements and actual query plans. Session startup and ingestion are outside the measured interval.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>قس أفعالًا متساوية النتائج</h2><p>CSV مقابل نسخة Delta رقم صفر؛ تهيئة لكل مسار وأربعة قياسات لكل مسار بترتيب متوازن. يحفظ الكود القياسات الخام وخطط الاستعلام الفعلية. إقلاع الجلسة والاستيعاب خارج الفترة المقاسة.</p></td></tr></tbody></table>

In [17]:
from masar.benchmark import benchmark
try:
    report = benchmark(spark, SOURCE, WORK, repetitions=4)
    print(json.dumps(report["expected_and_observed_aggregate"], indent=2))
    print(json.dumps(report["measurements"], indent=2))
    print("Plans:", report["plans"])
    print("Evidence:", (WORK / "reports/benchmark.json").relative_to(ROOT))
finally:
    spark.stop()

{
  "rows": 72,
  "nonnull_fares": 72,
  "fare_total": "1794.60"
}
{
  "csv": {
    "samples_s": [
      0.08590044000000319,
      0.10215454199999385,
      0.06415751900000544,
      0.09106827399999418
    ],
    "median_s": 0.08848435699999868,
    "min_s": 0.06415751900000544,
    "max_s": 0.10215454199999385
  },
  "delta_v0": {
    "samples_s": [
      0.5699540309999946,
      0.6305078480000077,
      0.5836660119999948,
      0.6033973130000021
    ],
    "median_s": 0.5935316624999984,
    "min_s": 0.5699540309999946,
    "max_s": 0.6305078480000077
  }
}
Plans: {'csv': 'reports/plans/csv.txt', 'delta_v0': 'reports/plans/delta_v0.txt'}
Evidence: outputs/day01_bronze_n6_lk_hh/reports/benchmark.json


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Interpret and retain</h2><p>Keep all samples and explain variability. Repeated reads may use OS/JVM/metadata caches; this is not a cold-cache or production test. Complete <a href="../../templates/BENCHMARKS.md">BENCHMARKS.md</a> and Lab 02 notes. See <a href="../../day01/COMPLETION.md">the Day 1 handoff</a>.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>فسر واحتفظ</h2><p>احفظ جميع العينات وفسر التفاوت. قد تستفيد القراءات المتكررة من ذاكرة نظام التشغيل وJVM والبيانات الوصفية؛ ليست تجربة ذاكرة فارغة أو اختبار إنتاج. أكمل <a href="../../templates/BENCHMARKS.md">BENCHMARKS.md</a> وملاحظات اللاب 02. راجع <a href="../../day01/COMPLETION.md">تسليم اليوم الأول</a>.</p></td></tr></tbody></table>